Hàng đợi hai đầu (double-ended queue)
Làm quen với cấu trúc dữ liệu Double-Ended Queue mà bạn sẽ dùng làm replay buffer.

Double-Ended Queue, hay deque, là một hàng đợi có dung lượng giới hạn, sẽ “quên” các phần tử cũ nhất khi phần tử mới được thêm vào theo thời gian. Điều này tạo ra một lớp trừu tượng rất phù hợp để làm replay buffer.

Hướng dẫn 1/3
35 XP
1
Thêm số nguyên 10 vào phía bên phải của Double-Ended Queue.

2
Đặt dung lượng tối đa là 5 bằng đối số maxlen; quan sát điều gì xảy ra khi bạn cố gắng thêm vượt quá dung lượng của hàng đợi.

3
Lấy ngẫu nhiên ba phần tử từ buffer.

In [ ]:
from collections import deque
buffer = deque(range(10))
print('Buffer initialized as:', buffer)
# Append 10 to the right of the buffer
buffer.____
print('Buffer after appending:', buffer)

In [ ]:
"""
from collections import deque
buffer = deque(range(10))
print('Buffer initialized as:', buffer)
# Append 10 to the right of the buffer
buffer.append(10)
print('Buffer after appending:', buffer)
"""

In [ ]:
from collections import deque
# Set a maximum capacity of 5
buffer = deque(range(10), ____)
print('Buffer initialized as:', buffer)
buffer.append(10)
print('Buffer after appending:', buffer)

In [ ]:
"""
from collections import deque
# Set a maximum capacity of 5
buffer = deque(range(10), maxlen=5)
print('Buffer initialized as:', buffer)
buffer.append(10)
print('Buffer after appending:', buffer)
"""

In [ ]:
from collections import deque
import random
buffer = deque(range(10), maxlen=5)
print('Buffer initialized as:', buffer)
# Draw three items from the buffer
batch = random.____(____, ____)
print('Random sample from buffer:', batch)

In [ ]:
"""
from collections import deque
import random
buffer = deque(range(10), maxlen=5)
print('Buffer initialized as:', buffer)
# Draw three items from the buffer
batch = random.sample(buffer, 3)
print('Random sample from buffer:', batch)
"""

Bộ đệm experience replay
Bây giờ bạn sẽ tạo cấu trúc dữ liệu để hỗ trợ Experience Replay, giúp agent học hiệu quả hơn rất nhiều.

Bộ đệm replay này cần hỗ trợ hai thao tác:

Lưu trữ các trải nghiệm vào bộ nhớ để lấy mẫu trong tương lai.
"Phát lại" một lô ngẫu nhiên các trải nghiệm trong quá khứ từ bộ nhớ của nó.
Vì dữ liệu được lấy mẫu từ bộ đệm sẽ được đưa vào một neural network, bộ đệm nên trả về các Tensor của torch cho tiện lợi.

Các module torch và random cùng lớp deque đã được import vào môi trường bài tập của bạn.

Hướng dẫn
100 XP
Hoàn thiện phương thức push() của ReplayBuffer bằng cách thêm experience_tuple vào bộ nhớ của buffer.
Trong phương thức sample(), rút một mẫu ngẫu nhiên kích thước batch_size từ self.memory.
Vẫn trong sample(), mẫu ban đầu là danh sách các bộ; đảm bảo chuyển nó thành một bộ các danh sách.
Chuyển actions_tensor về dạng (batch_size, 1) thay vì (batch_size).

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        experience_tuple = (state, action, reward, next_state, done)
        # Append experience_tuple to the memory buffer
        self.memory.____    
    def __len__(self):
        return len(self.memory)
    def sample(self, batch_size):
        # Draw a random sample of size batch_size
        batch = ____(____, ____)
        # Transform batch into a tuple of lists
        states, actions, rewards, next_states, dones = ____
        states_tensor = torch.tensor(states, dtype=torch.float32)
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
        next_states_tensor = torch.tensor(next_states, dtype=torch.float32)
        dones_tensor = torch.tensor(dones, dtype=torch.float32)
        # Ensure actions_tensor has shape (batch_size, 1)
        actions_tensor = torch.tensor(actions, dtype=torch.long).____
        return states_tensor, actions_tensor, rewards_tensor, next_states_tensor, dones_tensor

In [ ]:
"""
Bạn có thể thêm phần tử vào deque bằng phương thức .append(); trong trường hợp này, hãy áp dụng cho experience_tuple.
Dùng random.sample(sequence, k) để rút ngẫu nhiên k mẫu từ một sequence.
zip(*args) biến danh sách các bộ args thành một bộ gồm các danh sách.
Phương thức unsqueeze(dim) thêm một chiều có kích thước một tại chiều thứ dim; ở đây, dim là một.



class ReplayBuffer:
    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        experience_tuple = (state, action, reward, next_state, done)
        # Append experience_tuple to the memory buffer
        self.memory.append(experience_tuple)
    def __len__(self):
        return len(self.memory)
    def sample(self, batch_size):
        # Draw a random sample of size batch_size
        batch = random.sample(self.memory, batch_size)
        # Transform batch into a tuple of lists
        states, actions, rewards, next_states, dones = zip(*batch)
        states_tensor = torch.tensor(states, dtype=torch.float32)
        rewards_tensor = torch.tensor(rewards, dtype=torch.float32)
        next_states_tensor = torch.tensor(next_states, dtype=torch.float32)
        dones_tensor = torch.tensor(dones, dtype=torch.float32)
        # Ensure actions_tensor has shape (batch_size, 1)
        actions_tensor = torch.tensor(actions, dtype=torch.long).unsqueeze(1)
        return states_tensor, actions_tensor, rewards_tensor, next_states_tensor, dones_tensor
        
"""

DQN với experience replay
Bây giờ bạn sẽ thêm Experience Replay để huấn luyện một agent dùng Deep Q Network. Bạn sẽ tiếp tục sử dụng môi trường Lunar Lander như khi xây dựng Barebone DQN trước đó.

Ở mỗi bước, thay vì chỉ dùng kiến thức từ lần chuyển trạng thái mới nhất để cập nhật mạng, bộ đệm Experience Replay cho phép agent học từ một lô (batch) ngẫu nhiên các trải nghiệm gần đây. Điều này cải thiện đáng kể khả năng học về môi trường.

Các lớp QNetwork và ReplayBuffer từ các bài trước đã có sẵn và được khởi tạo như sau:

q_network = QNetwork(8, 4)
replay_buffer = ReplayBuffer(10000)
Hàm describe_episode() cũng có sẵn để mô tả các chỉ số ở cuối mỗi episode.

Hướng dẫn 1/2
50 XP
1
Đẩy trải nghiệm mới nhất vào Replay Buffer.


In [ ]:
for episode in range(10):
    state, info = env.reset()
    done = False
    step = 0
    episode_reward = 0
    while not done:
        step += 1
        q_values = q_network(state)        
        action = torch.argmax(q_values).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        # Store the latest experience in the replay buffer
        replay_buffer.____        
        state = next_state
        episode_reward += reward    
    describe_episode(episode, reward, episode_reward, step)

In [ ]:
"""
Dùng phương thức .push() của replay_buffer để đẩy một trải nghiệm vào bộ đệm; hàm này nhận các đối số state, action, reward, next_state và done.

for episode in range(10):
    state, info = env.reset()
    done = False
    step = 0
    episode_reward = 0
    while not done:
        step += 1
        q_values = q_network(state)        
        action = torch.argmax(q_values).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        # Store the latest experience in the replay buffer
        replay_buffer.push(state, action, reward, next_state, done)        
        state = next_state
        episode_reward += reward    
    describe_episode(episode, reward, episode_reward, step)
    
"""

Lấy mẫu một batch gồm 64 trải nghiệm từ replay buffer.
Tính các Q-value của trạng thái kế tiếp

In [ ]:
for episode in range(10):
    state, info = env.reset()
    done = False
    step = 0
    episode_reward = 0
    while not done:
        step += 1
        q_values = q_network(state)
        action = torch.argmax(q_values).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        # Store the latest experience in the replay buffer
        replay_buffer.push(state, action, reward, next_state, done)        
        if len(replay_buffer) >= batch_size:
            # Sample 64 experiences from the replay buffer
            states, actions, rewards, next_states, dones = ____.____(____)
            q_values = q_network(states).gather(1, actions).squeeze(1)
            # Obtain the next state Q-values
            next_state_q_values = q_network(next_states).____
            target_q_values = rewards + gamma * next_state_q_values * (1-dones)
            loss = nn.MSELoss()(target_q_values, q_values)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        state = next_state
        episode_reward += reward
    describe_episode(episode, reward, episode_reward, step)

In [ ]:
"""
Dùng phương thức .sample() của replay_buffer để lấy mẫu trải nghiệm; hàm này có một đối số là kích thước batch.
Q-value của trạng thái kế tiếp được lấy, với mỗi transition trong batch, bằng cách lấy Q-value lớn nhất trong số các hành động có thể ở trạng thái kế tiếp;
có thể thực hiện bằng .amax(dim), trong đó dim là chiều cần lấy max: ở đây dim=1.



for episode in range(10):
    state, info = env.reset()
    done = False
    step = 0
    episode_reward = 0
    while not done:
        step += 1
        q_values = q_network(state)        
        action = torch.argmax(q_values).item()
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        # Store the latest experience in the replay buffer
        replay_buffer.push(state, action, reward, next_state, done)        
        if len(replay_buffer) >= batch_size:
            # Sample 64 experiences from the replay buffer
            states, actions, rewards, next_states, dones = replay_buffer.sample(64)
            q_values = q_network(states).gather(1, actions).squeeze(1)
            # Obtain the next state Q-values
            next_state_q_values = q_network(next_states).amax(1)
            target_q_values = rewards + gamma * next_state_q_values * (1-dones)
            loss = nn.MSELoss()(target_q_values, q_values)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        state = next_state
        episode_reward += reward    
    describe_episode(episode, reward, episode_reward, step)
    
"""